# Word Embeddings

In this lab, we will see how to use pre-trained word embeddings with [`spacy`](https://spacy.io/) and [`gensim`](https://radimrehurek.com/gensim/).

We will then see how to train our own embeddings.

In [ ]:
%pip install gensim

In [ ]:
import itertools
import os
import pathlib
import sys
import re
import typing

import gensim.models
import gensim.models.phrases
import spacy

## Pre-trained Models With Spacy

To use language models in spacy, you first need to download them. There are embedding models for several languages:

- English
- German
- Greek
- Spanish
- French
- Italian
- Lithuanian
- Norwegian
- Dutch
- Portuguese

About forty other languages are planned (see [spacy language models](https://spacy.io/usage/models)).

We will work with the most complete model, the English one, which has embedding vectors for 20,000 words.

In [ ]:
!python -m spacy download en_core_web_sm
#!python -m spacy download fr_core_news_md
#!python -m spacy download de
# …

## Usage
Once the model is loaded, documents can be processed as follows:

In [ ]:
# Load the model
nlp = spacy.load('en_core_web_sm')

# Process the document with Spacy
doc = nlp("This is some text that I am processing with Spacy")

print("The dense vector of the 4th word")
print(doc[3].vector)

# The average dense vector of the sentence. Can be used for classification,
# for instance (even though it's a bad idea in absolute terms: the fasttext or
# transformers libraries would be better choices)
print("average vector")
print(doc.vector)

## Pre-trained Models With Gensim

Gensim doesn't come with its own dense word embedding vectors. You can easily use some, though, once downloaded.

Many sources of pre-trained embedding models are available for download.

- The most popular one is probably the 300-dimension Google News dataset model (see [Google News word2vec documentation](https://code.google.com/archive/p/word2vec/); [GoogleNews-vectors-negative300.bin.gz](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing))

- A source of varied models for many languages: [NLPL word embeddings repository](http://vectors.nlpl.eu/repository/)

- A model developed by Facebook that learns embeddings of word parts: [fasttext](https://fasttext.cc/docs/en/crawl-vectors.html)

- A repository for French: [Jean-Philippe Fauconnier](https://fauconnier.github.io/)

In what follows, we will use the French model from Jean-Philippe Fauconnier's repository (continuous bag of words, dimension 200, cutoff 100, no lemmatization, no POS tagging, no phrasing).

In [ ]:
!wget https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin

In [ ]:
!ls -l
model_file = "frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin"

## Using [`gensim.models.KeyedVectors`](https://radimrehurek.com/gensim/models/keyedvectors.html)

In [ ]:
# Load a word embeddings model
model = gensim.models.KeyedVectors.load_word2vec_format(model_file, binary=True)
print("number of words in the vocabulary: ",len(model.index_to_key))

In [ ]:
for i, word in enumerate(model.index_to_key):
  print(word)
  if i > 50:
    break

In [ ]:
# Turn a word into a vector
vector = model['facile']

# Turn a sentence into a list of vectors.
# The inputs must be tokenized.
# If a word isn't in the vocabulary, you'll get an error.
# Pay attention to the preprocessing you use compared to the model's
# (lemmatization, lowercasing, etc).
words = "Ceci est une phrase transformée à l'aide de Gensim".lower().split(' ')
vectors = [(word, model[word]) for word in words if word in model.key_to_index]
print(f"Transformed words: {' '.join(word for word, _ in vectors)}")

In [ ]:
model.most_similar('berlin')

In [ ]:
print(model.similarity('berlin', 'munich'))
print(model.similarity('berlin', 'paris'))

## Operations in the Semantic Space
Using the [`gensim.models.keyedvectors.KeyedVectors.similar_by_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector) function, define a function that performs the semantic translation seen in class.

This function takes 3 arguments `x`, `y` and `z` and answers the following question:

`x` is to `y` what `z` is to?

E.g.: *King* is to *Queen* what *Uncle* is to?

Answer: *Aunt*

Displaying several results is advised: it's simpler to code and makes it easier to find what you're looking for (the first answer is often `z`…)

In [ ]:
def semantic_analogy(x, y, z):
  # Your code here
  return "Tante"

### Solution

In [ ]:
def semantic_analogy(x, y, z):
  return model.similar_by_vector(model[y] - model[x] + model[z])

## Your Turn
Use this function on examples that come to mind.

In [ ]:
semantic_analogy("homme", "femme", "chien")

## Training Word Embeddings

Learning dense representations on your own data isn't complicated. It can be particularly suited to specific document corpora (notarial deeds, ancient languages, …)

### The 20 Newsgroups Corpus

This corpus is made of about 20,000 "posts" split into 20 themes (since they come from 20 different newsgroups).

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

In [ ]:
!ls 20news-bydate-test/

In [ ]:
# Each folder is a newsgroup, then one article per file
contents = [p.read_text(encoding="latin-1")
            for p in pathlib.Path(".").glob("20news-bydate-*/*/*")]


print(f"{len(contents)} texts retrieved")

In [ ]:
print(contents[1])

In [ ]:
# Remove punctuation and tokenize.
def preprocess(content: str) -> typing.List[str]:
  offset = content.find('\n\n')
  if offset > 0:
    content = content[offset + 2:]
  sentences = (re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
               for line in content.splitlines()
               if not line.endswith("writes:"))
  return list(itertools.chain.from_iterable(sentences))


texts = [preprocess(content) for content in contents]

In [ ]:
print(texts[0])

## Detecting Common Noun Phrases With Gensim Phraser

Some word bigrams are very common (e.g. New York). It's often useful to detect these bigrams to treat them as "words" of the vocabulary.

That's what the Phraser object is for: it detects them and transforms our corpus accordingly.

In [ ]:
# Build the common noun phrases
phrases = gensim.models.phrases.Phrases(
    texts, connector_words=gensim.models.phrases.ENGLISH_CONNECTOR_WORDS)

# The Phraser object now lets us transform our corpus
phraser = gensim.models.phrases.Phraser(phrases)
phrased_texts = list(phraser[texts])
len(phrased_texts)

In [ ]:
trainable_model = gensim.models.Word2Vec(
    phrased_texts,
    min_count=3,      # Threshold above which words aren't ignored
    vector_size=200,  # Word embedding dimension
    workers=2,        # Number of threads
    window=5,         # Context window size
    epochs=30)        # Number of iterations
model = trainable_model.wv  # Get the KeyedVectors instance after
                            # training

In [ ]:
model.most_similar("New_York")

In [ ]:
semantic_analogy("wind", "fly", "water")